# 第 14 章 · SWE-bench 批量评测

**这一章你会得到什么**：看清一个 Agent 评测框架的骨架——如何把一个 benchmark 实例变成一次 Agent run，如何并发跑、如何把结果写成可评分的 predictions 文件。**本章不跑真实 docker/数据集**，只跑离线的纯函数。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/run/benchmarks/swebench.py` **L68–76** — `get_swebench_docker_image_name`（实例→镜像名）
- `src/minisweagent/run/benchmarks/swebench.py` **L122–177** — `process_instance`（单实例：组装+run+存轨迹+写preds，try/finally）
- `src/minisweagent/run/benchmarks/swebench.py` **L180–197** — `filter_instances`（过滤/切片）
- `src/minisweagent/run/benchmarks/swebench.py` **L97–108** — `update_preds_file`（写 preds.json）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [ ]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

In [ ]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 概念：一次评测 = 一堆“单实例 run”的并发编排

`swebench.py` 的 `main()` 做四件事：加载数据集 -> 过滤/切片 -> 线程池并发 `process_instance` -> 每个实例存 trajectory + 更新 preds.json。
每个 `process_instance` 本质就是第 13 章那套组装 + `agent.run(task)`，只是环境换成对应实例的 docker 镜像。

In [ ]:
show_source("src/minisweagent/run/benchmarks/swebench.py", 122, 177)

## 实验 1：实例 ID 如何映射到 docker 镜像名

注意 `__` 会被替换成 `_1776_`（docker 不允许双下划线）。

In [ ]:
from minisweagent.run.benchmarks.swebench import get_swebench_docker_image_name
print(get_swebench_docker_image_name({"instance_id": "django__django-12345"}))
print(get_swebench_docker_image_name({"instance_id": "astropy__astropy-7008"}))
print(get_swebench_docker_image_name({"instance_id": "x", "image_name": "custom/image:tag"}))

## 实验 2：过滤与切片

`filter_instances` 用正则筛 instance_id，再按 `start:stop` 切片。

In [ ]:
from minisweagent.run.benchmarks.swebench import filter_instances
insts = [{"instance_id": f"django__django-{i}"} for i in range(4)] + \
        [{"instance_id": f"flask__flask-{i}"} for i in range(3)]
only_django = filter_instances(insts, filter_spec="django.*")
print("正则筛 django:", [i["instance_id"] for i in only_django])
first_two = filter_instances(insts, filter_spec="", slice_spec="0:2")
print("切片前两个:", [i["instance_id"] for i in first_two])

## 实验 3：predictions 文件的读写

评测最终产出一个 `preds.json`：instance_id -> {model_patch, ...}。这是喂给官方评分器的接口。
我们在临时目录里写一条、读回来。

In [ ]:
import json, tempfile
from pathlib import Path
from minisweagent.run.benchmarks.swebench import update_preds_file
d = Path(tempfile.mkdtemp())
preds = d / "preds.json"
update_preds_file(preds, "django__django-12345", "gpt-4o", "diff --git a/x b/x ...")
update_preds_file(preds, "flask__flask-1", "gpt-4o", "diff --git a/y b/y ...")
loaded = json.loads(preds.read_text())
print("实例数:", len(loaded))
print("一条记录:", loaded["flask__flask-1"])
# 清理
for p in d.iterdir(): p.unlink()
d.rmdir()

## 观察点
- `process_instance` 用 `try/except/finally`：**无论成功失败都存 trajectory + 写 preds**——评测最怕“跑了半天没留下记录”。这和第 4/7 章“finally 里 save”一脉相承。
- 每个实例的失败被隔离（异常记进该实例的 exit_status），不会拖垮整个 batch。
- 结果落地成标准 `preds.json`：**Agent 的“分数”不由框架自己判，而是产出标准格式交给外部评分器**。这是可信评测的关键边界。

## 动手：给一个不带 image_name 的实例，手推它的镜像名
先在心里推 `"scikit-learn__scikit-learn-999"` 会变成什么镜像名，再运行核对。

In [ ]:
print(get_swebench_docker_image_name({"instance_id": "scikit-learn__scikit-learn-999"}))

## 闭卷检查
1. 一次评测里，单个实例的处理流程是什么？
2. 为什么存 trajectory / 写 preds 放在 finally？
3. 为什么框架产出标准 preds.json 而不自己判分？